Chapter 2 of Tanimura's SQL for Data Analysis covers Data Preparation—specifically cleaning, reformatting, and transforming dirty untamed inputs into clean analytical sets.

To an engineer or quantitative analyst, Chapter 2 reads like basic string manipulation and type-casting. However, the core takeaway isn't learning how to format a string—it's understanding why relational databases require set-based data transformation before mathematical operations can begin.

The 4 Core Takeaways of Chapter 2

1. The Core Philosophy: "Database Prep First, Analytics Second"
   In Python/Pandas, you often load raw strings and parse dates vectorially in memory (pd.to_datetime()). In large-scale SQL data warehouses, doing raw transformations inside Python memory crashes kernels or creates network bandwidth bottlenecks.

Takeaway: Data cleaning must happen inside the database engine via set operations so downstream aggregations execute at hardware speed.

2. Type Enforcement & Casting
   Raw incoming data (especially from CSVs or event streams) often defaults to generic string/text types (VARCHAR). SQL requires explicit type discipline before any mathematical or temporal operator can function.

Key Syntax: CAST(column AS TYPE) or the shorthand column::TYPE (supported in Postgres and DuckDB).

Common Types: DATE, TIMESTAMP, NUMERIC / DECIMAL(p,s), INTEGER, BOOLEAN.

3. String Parsing & Structural Standardizations
   Uncleaned text makes categorical aggregations (GROUP BY) fail or produce false duplicates (e.g., 'Retail', 'retail ', 'RETAIL').

Case & Trim: LOWER(), UPPER(), TRIM() remove trailing whitespace and standardize casing.

Substrings & Slicing: Extracting fixed codes or dates using SUBSTRING(string FROM start FOR length) or LEFT() / RIGHT().

Concatenation: Combining fields into composite keys using CONCAT() or the standard SQL operator ||.

4. Handling Missing Data (NULL Logic)
   In SQL, NULL is not zero or an empty string—it represents unknown state (three-valued logic: TRUE, FALSE, UNKNOWN).

Arithmetic operations with NULL always evaluate to NULL (e.g., 10 + NULL = NULL).

Key Function: COALESCE(val1, val2, val3, ...) returns the first non-null value in the sequence—essential for setting default baseline values in calculations.


In [ ]:
-- The "Data Extractor" Baseline:
SELECT date, entity_id, factor_1, factor_2
FROM database.data_table
WHERE date >= '2020-01-01' AND status = 'ACTIVE';

In [ ]:
# CHAPTER 2: Preparing data for analysis

import duckdb
import pandas as pd

conn = duckdb.connect(database=":memory:")

# Create a mock raw table demonstrating typical dirty data inputs
conn.execute("""
CREATE TABLE raw_transactions (
    raw_date VARCHAR,
    store_code VARCHAR,
    raw_amount VARCHAR
);

INSERT INTO raw_transactions VALUES
    ('2026-01-15', '  store_001 ', '$150.50'),
    ('2026-01-16', 'STORE_001', NULL),
    ('2026-02-01', 'Store_002  ', '$89.00');
""")

# Chapter 2 SQL Cleaning Transformations
cleaned_df = conn.execute("""
SELECT
    -- 1. Date Casting
    CAST(raw_date AS DATE) AS transaction_date,

    -- 2. String Standardization (Trim & Lowercase)
    LOWER(TRIM(store_code)) AS clean_store_id,

    -- 3. String Stripping & Numeric Casting
    CAST(REPLACE(raw_amount, '$', '') AS NUMERIC(10,2)) AS raw_amount_clean,

    -- 4. NULL Handling via COALESCE (Default missing sales to 0.00)
    COALESCE(CAST(REPLACE(raw_amount, '$', '') AS NUMERIC(10,2)), 0.00) AS final_sales_amount
FROM raw_transactions;
""").df()

cleaned_df

| Concept             | Problem Solved                                    | Primary SQL Functions          |
| :------------------ | :------------------------------------------------ | :----------------------------- |
| **Type Casting**    | Strings won't allow math or date arithmetic       | `CAST(x AS TYPE)`, `x::TYPE`   |
| **String Cleaning** | Whitespace & inconsistent casing break `GROUP BY` | `TRIM()`, `LOWER()`, `UPPER()` |
| **Substrings**      | Slicing specific codes out of unstructured text   | `SUBSTRING()`, `SPLIT_PART()`  |
| **Missing Values**  | `NULL` propagating through arithmetic equations   | `COALESCE(val, default_val)`   |
